In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
customer = spark.read.format("parquet").load("abfss://bronze@intechaccstorage.dfs.core.windows.net/Customer")
customer.limit(5).display()

CustomerID,NameStyle,Title,FirstName,MiddleName,LastName,Suffix,CompanyName,SalesPerson,EmailAddress,Phone,PasswordHash,PasswordSalt,rowguid,ModifiedDate,_rescued_data
1,False,Mr.,Orlando,N.,Gee,null,A Bike Store,adventure-works\pamela0,orlando0@adventure-works.com,245-555-0173,L/Rlwxzp4w7RWmEgXX+/A7cXaePEPcp+KwQhl2fJL7w=,1KjXYs4=,3f5ae95e-b87d-4aed-95b4-c3797afcb74f,2005-08-01 00:00:00.0000000,null
2,False,Mr.,Keith,null,Harris,null,Progressive Sports,adventure-works\david8,keith0@adventure-works.com,170-555-0127,YPdtRdvqeAhj6wyxEsFdshBDNXxkCXn+CRgbvJItknw=,fs1ZGhY=,e552f657-a9af-4a7d-a645-c429d6e02491,2006-08-01 00:00:00.0000000,null
3,False,Ms.,Donna,F.,Carreras,null,Advanced Bike Components,adventure-works\jillian0,donna0@adventure-works.com,279-555-0130,LNoK27abGQo48gGue3EBV/UrlYSToV0/s87dCRV7uJk=,YTNH5Rw=,130774b1-db21-4ef3-98c8-c104bcd6ed6d,2005-09-01 00:00:00.0000000,null
4,False,Ms.,Janet,M.,Gates,null,Modular Cycle Systems,adventure-works\jillian0,janet1@adventure-works.com,710-555-0173,ElzTpSNbUW1Ut+L5cWlfR7MF6nBZia8WpmGaQPjLOJA=,nm7D5e4=,ff862851-1daa-4044-be7c-3e85583c054d,2006-07-01 00:00:00.0000000,null
5,False,Mr.,Lucy,null,Harrington,null,Metropolitan Sports Supply,adventure-works\shu0,lucy0@adventure-works.com,828-555-0186,KJqV15wsX3PG8TS5GSddp6LFFVdd3CoRftZM/tP0+R4=,cNFKU4w=,83905bdc-6f5e-4f71-b162-c98da069f38a,2006-09-01 00:00:00.0000000,null


In [0]:
customer = customer.withColumn("FirstName", \
    trim(concat_ws(" ", col("FirstName"), col("MiddleName"), col("LastName"))))\
        .withColumn("SalesPerson", split(col("SalesPerson"), "\\\\")[1])\
        .withColumn("ModifiedDate", to_date(col("ModifiedDate"))) 

customer = customer.withColumnRenamed('SalesPerson', 'Sales_person')\
                    .withColumnRenamed('ModifiedDate', 'Modified_date')\
                    .withColumnRenamed('CustomerID', 'Customer_id')\
                    .withColumnRenamed('CompanyName', 'Company_name')\
                    .withColumnRenamed('EmailAddress', 'Email_address')\
                    .withColumnRenamed('FirstName', 'Full_Name')


In [0]:
#drop 
customer = customer.drop("PasswordHash", "PasswordSalt", "NameStyle", "rowguid", "_rescued_data", "MiddleName", "LastName", "Suffix")
customer.limit(5).display()

Customer_id,Title,Full_Name,Company_name,Sales_person,Email_address,Phone,Modified_date
1,Mr.,Orlando N. Gee,A Bike Store,pamela0,orlando0@adventure-works.com,245-555-0173,2005-08-01
2,Mr.,Keith Harris,Progressive Sports,david8,keith0@adventure-works.com,170-555-0127,2006-08-01
3,Ms.,Donna F. Carreras,Advanced Bike Components,jillian0,donna0@adventure-works.com,279-555-0130,2005-09-01
4,Ms.,Janet M. Gates,Modular Cycle Systems,jillian0,janet1@adventure-works.com,710-555-0173,2006-07-01
5,Mr.,Lucy Harrington,Metropolitan Sports Supply,shu0,lucy0@adventure-works.com,828-555-0186,2006-09-01


In [0]:
customer.columns

['Customer_id',
 'Title',
 'Full_Name',
 'Company_name',
 'Sales_person',
 'Email_address',
 'Phone',
 'Modified_date']

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricksintech.silver.customer
(
    Customer_id INTEGER,
    Title STRING,
    Full_Name STRING,
    Company_name STRING,
    Sales_person STRING,
    Email_address STRING,
    Phone STRING,
    Modified_date DATE
) 
USING DELTA
LOCATION 'abfss://silver@intechaccstorage.dfs.core.windows.net/customer';

In [0]:
# Reference target Delta table
silver_table = DeltaTable.forName(spark, "databricksintech.silver.customer")

# Execute the Merge (Upsert)
silver_table.alias("target").merge(
    source = customer.alias("source"),
    condition = (
        "target.Customer_id = source.Customer_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
